# Codex Configuration

This notebook inspects the current environment (network reachability, local LLM
services, MCP servers) and reconciles it against `~/.codex/config.toml` and
`.codex/config.toml` in the workspace.

It will:

1. Probe the environment — host gateway, LM Studio (direct + via the namespace-tools shim), Ollama, MCP servers, the `codex` and `codex-acp` binaries.
2. Load and pretty-print the current Codex configs (user-scope + project-scope) plus the active model catalog.
3. Diagnose missing or conflicting entries.
4. *On request* patch the configs in place using `tomlkit`, so comments and any benign settings you've added by hand survive untouched.

The intended default profile after this notebook is one that is known to work
in the **current** environment state — for this container that means
LM Studio + the Responses namespace-tools shim + `qwen/qwen3-coder-next`.

Background reading:
[docs/codex-lmstudio-shim.md](docs/codex-lmstudio-shim.md),
[docs/mcp-config-and-slash-commands.md](docs/mcp-config-and-slash-commands.md).

## 0. Prerequisites

We rely on `tomlkit` for round-trip-preserving TOML edits. It is already pulled
in by Codex / the kernel, but install on demand if missing.

In [1]:
import importlib, subprocess, sys

for pkg in ("tomlkit",):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", pkg])

import json, os, shutil, socket, urllib.error, urllib.request
from pathlib import Path
from textwrap import indent

import tomlkit

HOME = Path.home()
USER_CONFIG = HOME / ".codex" / "config.toml"
USER_CONFIG_BACKUP_TEMPLATE = ".codex/config.toml.bak.{ts}"
PROJECT_ROOT = Path("/workspaces/agent-client-kernel")
PROJECT_CONFIG = PROJECT_ROOT / ".codex" / "config.toml"
CATALOG_DIR = HOME / ".codex" / "model-catalogs"
CATALOG_FILE = CATALOG_DIR / "local-lmstudio.json"

SHIM_PORT = int(os.environ.get("ACP_LMSTUDIO_SHIM_PORT") or os.environ.get("SHIM_PORT") or 18234)
SHIM_OFF = os.environ.get("ACP_LMSTUDIO_SHIM", "").lower() == "off"
HOST_GATEWAY = os.environ.get("HOST_GATEWAY_IP") or "host.docker.internal"
EXPLICIT_TARGET = os.environ.get("ACP_LMSTUDIO_SHIM_TARGET") or os.environ.get("SHIM_TARGET")
LMSTUDIO_DIRECT = EXPLICIT_TARGET or f"http://{HOST_GATEWAY}:1234"
LMSTUDIO_VIA_SHIM = f"http://127.0.0.1:{SHIM_PORT}/v1"

print("User config :", USER_CONFIG, "(exists)" if USER_CONFIG.exists() else "(MISSING)")
print("Project cfg :", PROJECT_CONFIG, "(exists)" if PROJECT_CONFIG.exists() else "(missing)")
print("Model catalog:", CATALOG_FILE, "(exists)" if CATALOG_FILE.exists() else "(MISSING)")
print("LM Studio upstream :", LMSTUDIO_DIRECT)
print("LM Studio via shim :", LMSTUDIO_VIA_SHIM, "(shim disabled)" if SHIM_OFF else "")

User config : /home/jovyan/.codex/config.toml (exists)
Project cfg : /workspaces/agent-client-kernel/.codex/config.toml (exists)
Model catalog: /home/jovyan/.codex/model-catalogs/local-lmstudio.json (exists)
LM Studio upstream : http://host.docker.internal:1234
LM Studio via shim : http://127.0.0.1:18234/v1 


## 1. Probe the environment

In [2]:
def tcp_open(host: str, port: int, timeout: float = 1.0) -> bool:
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False

def http_json(url: str, timeout: float = 3.0):
    req = urllib.request.Request(url, headers={"Accept": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            return resp.status, json.loads(resp.read().decode("utf-8", "replace"))
    except urllib.error.HTTPError as e:
        return e.code, None
    except (urllib.error.URLError, TimeoutError, ConnectionError, json.JSONDecodeError) as e:
        return None, str(e)

def which(cmd: str):
    return shutil.which(cmd)

env_facts = {}

# --- Network: host gateway reachability ---
gw_host, gw_port = HOST_GATEWAY, 1234
env_facts["host_gateway"] = HOST_GATEWAY
env_facts["host_gateway_resolves"] = False
try:
    socket.gethostbyname(HOST_GATEWAY)
    env_facts["host_gateway_resolves"] = True
except OSError:
    pass

# --- LM Studio (direct) ---
env_facts["lmstudio_direct_url"] = LMSTUDIO_DIRECT
env_facts["lmstudio_direct_reachable"] = tcp_open(gw_host, gw_port)
status, payload = http_json(f"{LMSTUDIO_DIRECT}/v1/models")
env_facts["lmstudio_direct_models_status"] = status
if isinstance(payload, dict):
    env_facts["lmstudio_models"] = [m.get("id") for m in payload.get("data", []) if isinstance(m, dict)]
else:
    env_facts["lmstudio_models"] = []
    env_facts["lmstudio_direct_error"] = payload

# --- LM Studio via shim ---
env_facts["lmstudio_shim_port"] = SHIM_PORT
env_facts["lmstudio_shim_disabled_by_env"] = SHIM_OFF
env_facts["lmstudio_shim_running"] = tcp_open("127.0.0.1", SHIM_PORT)
if env_facts["lmstudio_shim_running"]:
    status, payload = http_json(f"{LMSTUDIO_VIA_SHIM}/models")
    env_facts["lmstudio_shim_models_status"] = status

# --- Ollama (optional) ---
env_facts["ollama_reachable"] = tcp_open("127.0.0.1", 11434) or tcp_open(gw_host, 11434)
if env_facts["ollama_reachable"]:
    for url in ("http://127.0.0.1:11434/api/tags", f"http://{gw_host}:11434/api/tags"):
        status, payload = http_json(url)
        if isinstance(payload, dict):
            env_facts["ollama_url"] = url.rsplit("/api", 1)[0]
            env_facts["ollama_models"] = [m.get("name") for m in payload.get("models", [])]
            break

# --- Tooling ---
env_facts["codex_bin"] = which("codex")
env_facts["codex_acp_bin"] = which("codex-acp") or os.environ.get("ACP_AGENT_COMMAND")
env_facts["uvx_bin"] = which("uvx")
env_facts["jupyter_mcp_via_uvx"] = bool(env_facts["uvx_bin"])  # uvx fetches on demand
try:
    import paper_search_mcp  # noqa: F401
    env_facts["paper_search_mcp_installed"] = True
except ImportError:
    env_facts["paper_search_mcp_installed"] = False

# --- Jupyter server (for jupyter-mcp-server target) ---
env_facts["jupyter_lab_reachable"] = tcp_open("127.0.0.1", int(os.environ.get("JUPYTER_PORT", "8888")))

print(json.dumps(env_facts, indent=2, default=str))

{
  "host_gateway": "host.docker.internal",
  "host_gateway_resolves": true,
  "lmstudio_direct_url": "http://host.docker.internal:1234",
  "lmstudio_direct_reachable": true,
  "lmstudio_direct_models_status": 200,
  "lmstudio_models": [
    "qwen/qwen3-coder-next",
    "google/gemma-4-31b",
    "unsloth/gemma-4-31b-it",
    "gemma-4-31b",
    "f2llm-v2",
    "qwen3.5-122b-a10b",
    "qwen3.5-27b",
    "qwen3.5-35b-a3b",
    "qwen/qwen3-next-80b",
    "llama-3.2-1b-instruct",
    "qwen2.5-0.5b-instruct-mlx",
    "glm-4.7-reap-218b-a32b",
    "glm-4.7-flash",
    "zai-org/glm-4.7-flash",
    "nomic-ai/text-embedding-nomic-embed-text-v1.5",
    "devstral-2-123b-instruct-2512",
    "minimax/minimax-m2",
    "s3dev-ai/text-embedding-nomic-embed-text-v1.5",
    "nomicai-modernbert-embed-base",
    "nvidia/nemotron-3-nano",
    "mistralai/devstral-small-2-2512",
    "deepswe-preview-mlx",
    "devstral-small-2507-mlx",
    "openai/gpt-oss-120b",
    "google/gemma-3n-e4b",
    "qwen/qwen3-cod

## 2. Load Codex configuration files

We parse with `tomlkit` so the round-trip preserves comments and any extra keys
you may have added.

In [3]:
def load_toml(path: Path):
    if not path.exists():
        return None
    return tomlkit.parse(path.read_text())

user_doc = load_toml(USER_CONFIG)
project_doc = load_toml(PROJECT_CONFIG)

def summarize(doc, label):
    print(f"--- {label} ---")
    if doc is None:
        print("(file does not exist)")
        return
    print("top-level keys:", list(doc.keys()))
    if "profile" in doc:
        print("  profile =", doc["profile"])
    if "profiles" in doc:
        print("  profiles:", list(doc["profiles"].keys()))
    if "model_providers" in doc:
        for name, prov in doc["model_providers"].items():
            print(f"  model_providers.{name}.base_url = {prov.get('base_url')}")
    if "mcp_servers" in doc:
        print("  mcp_servers:", list(doc["mcp_servers"].keys()))
    print()

summarize(user_doc, "~/.codex/config.toml")
summarize(project_doc, str(PROJECT_CONFIG))

catalog = None
if CATALOG_FILE.exists():
    catalog = json.loads(CATALOG_FILE.read_text())
    print("--- model catalog ---")
    for m in catalog.get("models", []):
        print(f"  {m['slug']:40s}  ctx={m.get('context_window')}")

--- ~/.codex/config.toml ---
top-level keys: ['profile', 'profiles', 'model_providers']
  profile = lmstudio-qwen3-coder
  profiles: ['lmstudio-qwen3-coder', 'lmstudio-qwen35', 'lmstudio-gemma4']
  model_providers.local_lmstudio.base_url = http://127.0.0.1:18234/v1

--- /workspaces/agent-client-kernel/.codex/config.toml ---
top-level keys: ['profile', 'mcp_servers', 'profiles', 'model_providers', 'projects']
  profile = lmstudio-qwen3-coder
  profiles: ['lmstudio-qwen3-coder', 'lmstudio-qwen35', 'lmstudio-gemma4']
  model_providers.local_lmstudio.base_url = http://127.0.0.1:18234/v1
  mcp_servers: ['jupyter', 'paper-search']

--- model catalog ---
  openai/gpt-oss-120b                       ctx=131072
  qwen3.5-27b                               ctx=32768
  qwen/qwen3-coder-next                     ctx=262144
  google/gemma-4-26b-a4b                    ctx=32768


## 3. Diagnose

We build a list of `(severity, message, fix)` entries. `severity` is one of
`error` (must fix for current env to work), `warn` (likely wrong but not fatal),
or `info`. The `fix` is a callable applied later if the user opts in.

In [4]:
issues: list[tuple[str, str, object]] = []  # (severity, message, fix-key)
fixes: dict[str, object] = {}

# Which profile *should* be the default given the live environment?
PREFERRED_MODEL = "qwen/qwen3-coder-next"
PREFERRED_PROFILE = "lmstudio-qwen3-coder"
PREFERRED_PROVIDER = "local_lmstudio"

lm_models = set(env_facts.get("lmstudio_models") or [])
shim_up = env_facts["lmstudio_shim_running"] and not SHIM_OFF
lmstudio_up = bool(env_facts["lmstudio_direct_reachable"])
qwen_loaded = PREFERRED_MODEL in lm_models

if not lmstudio_up:
    issues.append(("error", f"LM Studio not reachable at {LMSTUDIO_DIRECT}. Start it on the host (port 1234) before using local profiles.", None))
elif not lm_models:
    issues.append(("warn", "LM Studio reachable but /v1/models returned no models. Load a model in LM Studio.", None))
elif not qwen_loaded:
    issues.append(("warn", f"Preferred model {PREFERRED_MODEL!r} not loaded in LM Studio. Available: {sorted(lm_models)}", None))

if not SHIM_OFF and not shim_up:
    issues.append(("error", f"LM Studio namespace-tools shim is not listening on 127.0.0.1:{SHIM_PORT}. Run scripts/start-lmstudio-shim.sh or `python -m agent_client_kernel.lmstudio_shim`.", None))

# --- User config (must hold profile/profiles/model_providers in Codex 0.130+) ---
if user_doc is None:
    issues.append(("error", f"{USER_CONFIG} is missing. Codex 0.130+ requires profile/profiles/model_providers at user scope.", "create_user_config"))
    fixes["create_user_config"] = ("create", USER_CONFIG)
else:
    expected_base_url = LMSTUDIO_VIA_SHIM if not SHIM_OFF else f"{LMSTUDIO_DIRECT}/v1"
    prov = user_doc.get("model_providers", {}).get(PREFERRED_PROVIDER)
    if prov is None:
        issues.append(("error", f"[model_providers.{PREFERRED_PROVIDER}] missing from user config.", "set_provider"))
        fixes["set_provider"] = ("set_provider", expected_base_url)
    else:
        cur = prov.get("base_url")
        if cur != expected_base_url:
            issues.append(("warn", f"model_providers.{PREFERRED_PROVIDER}.base_url is {cur!r}; environment suggests {expected_base_url!r}.", "set_provider_url"))
            fixes["set_provider_url"] = ("set_provider_url", expected_base_url)
        if prov.get("wire_api") != "responses":
            issues.append(("warn", f"model_providers.{PREFERRED_PROVIDER}.wire_api should be 'responses' for LM Studio.", "set_wire_api"))
            fixes["set_wire_api"] = ("set_wire_api", "responses")

    profiles = user_doc.get("profiles", {})
    if PREFERRED_PROFILE not in profiles:
        issues.append(("error", f"profile {PREFERRED_PROFILE!r} not defined.", "add_profile"))
        fixes["add_profile"] = ("add_profile", PREFERRED_PROFILE)
    else:
        prof = profiles[PREFERRED_PROFILE]
        if prof.get("model") != PREFERRED_MODEL:
            issues.append(("warn", f"profile {PREFERRED_PROFILE}.model={prof.get('model')!r}, expected {PREFERRED_MODEL!r}.", "fix_profile_model"))
            fixes["fix_profile_model"] = ("fix_profile_model", PREFERRED_MODEL)
        if prof.get("model_provider") != PREFERRED_PROVIDER:
            issues.append(("warn", f"profile {PREFERRED_PROFILE}.model_provider={prof.get('model_provider')!r}, expected {PREFERRED_PROVIDER!r}.", "fix_profile_provider"))
            fixes["fix_profile_provider"] = ("fix_profile_provider", PREFERRED_PROVIDER)

    # Pick a working default profile based on live env state.
    desired_default = PREFERRED_PROFILE if qwen_loaded else None
    if desired_default is None:
        # Fall back to any profile whose model is currently loaded.
        for name, prof in profiles.items():
            if prof.get("model") in lm_models and prof.get("model_provider") == PREFERRED_PROVIDER:
                desired_default = name
                break
    if desired_default and user_doc.get("profile") != desired_default:
        issues.append(("warn", f"top-level `profile` is {user_doc.get('profile')!r}; current env best matches {desired_default!r}.", "set_default_profile"))
        fixes["set_default_profile"] = ("set_default_profile", desired_default)

# --- Model catalog ---
if not CATALOG_FILE.exists():
    issues.append(("error", f"Model catalog {CATALOG_FILE} missing — profiles reference it via model_catalog_json.", "create_catalog"))
    fixes["create_catalog"] = ("create_catalog", None)
elif catalog is not None:
    slugs = {m["slug"] for m in catalog.get("models", [])}
    if PREFERRED_MODEL not in slugs:
        issues.append(("warn", f"Catalog has no entry for {PREFERRED_MODEL!r}.", "add_catalog_entry"))
        fixes["add_catalog_entry"] = ("add_catalog_entry", PREFERRED_MODEL)

# --- Project config: MCP servers should not duplicate keys Codex drops at project scope ---
if project_doc is not None:
    for forbidden in ("profile", "profiles", "model_providers"):
        if forbidden in project_doc:
            issues.append(("warn", f"project .codex/config.toml has `{forbidden}` — Codex 0.130+ ignores it at project scope. Safe to leave, but it lives in user config too.", None))
    if "mcp_servers" in project_doc:
        if "jupyter" in project_doc["mcp_servers"] and not env_facts["jupyter_lab_reachable"]:
            issues.append(("warn", "mcp_servers.jupyter is configured but Jupyter Lab is not listening on 127.0.0.1:8888.", None))
        if "paper-search" in project_doc["mcp_servers"] and not env_facts["paper_search_mcp_installed"]:
            issues.append(("warn", "mcp_servers.paper-search is configured but `paper_search_mcp` is not importable. `pip install paper-search-mcp` if you want it.", None))
    if not env_facts["uvx_bin"] and project_doc.get("mcp_servers", {}).get("jupyter", {}).get("command") == "uvx":
        issues.append(("error", "mcp_servers.jupyter uses `uvx` but uvx is not on PATH.", None))

# --- Report ---
sev_order = {"error": 0, "warn": 1, "info": 2}
issues.sort(key=lambda x: sev_order.get(x[0], 99))
if not issues:
    print("All checks passed. Current Codex config matches the live environment.")
else:
    for sev, msg, key in issues:
        tag = {"error": "[ERROR]", "warn": "[WARN] ", "info": "[INFO] "}[sev]
        suffix = f"  (fix: {key})" if key else ""
        print(f"{tag} {msg}{suffix}")

[WARN]  project .codex/config.toml has `profile` — Codex 0.130+ ignores it at project scope. Safe to leave, but it lives in user config too.
[WARN]  project .codex/config.toml has `profiles` — Codex 0.130+ ignores it at project scope. Safe to leave, but it lives in user config too.
[WARN]  project .codex/config.toml has `model_providers` — Codex 0.130+ ignores it at project scope. Safe to leave, but it lives in user config too.


## 4. Apply fixes

Set `APPLY = True` and re-run this cell to write the corrections back. The user
config is patched in place via `tomlkit`, so comments and any unrelated keys are
preserved. A timestamped backup is written before each change.

Only the issues that the previous cell tagged with a `fix:` key are applied —
benign extra settings are left alone.

In [5]:
import datetime as _dt

APPLY = False  # flip to True to write changes

USER_CONFIG_TEMPLATE = '''# Codex user-level config — managed by codex-configuration.ipynb.
# Holds the keys Codex 0.130+ refuses to read from a project-local config
# (profile, profiles, model_providers). Project-local overrides
# (mcp_servers, sandbox_mode, project trust) live in .codex/config.toml
# alongside this file and are merged in by Codex at load time.

profile = "{profile}"
'''

PROFILE_DEFAULTS = {
    "lmstudio-qwen3-coder": {
        "model": "qwen/qwen3-coder-next",
        "model_provider": "local_lmstudio",
        "model_catalog_json": str(CATALOG_FILE),
        "model_context_window": 262144,
        "model_auto_compact_token_limit": 240000,
        "model_reasoning_summary": "none",
        "model_supports_reasoning_summaries": False,
        "sandbox_mode": "danger-full-access",
        "personality": "pragmatic",
    },
}

PROVIDER_DEFAULTS = {
    "local_lmstudio": {
        "name": "LM Studio",
        "base_url": LMSTUDIO_VIA_SHIM if not SHIM_OFF else f"{LMSTUDIO_DIRECT}/v1",
        "wire_api": "responses",
        "requires_openai_auth": False,
        "supports_websockets": False,
        "stream_idle_timeout_ms": 300000,
        "request_max_retries": 1,
        "stream_max_retries": 0,
    },
}

def _backup(path: Path) -> Path:
    ts = _dt.datetime.now().strftime("%Y%m%d-%H%M%S")
    bak = path.with_name(path.name + f".bak.{ts}")
    bak.write_bytes(path.read_bytes())
    return bak

def _ensure_table(doc, key):
    if key not in doc:
        doc[key] = tomlkit.table()
    return doc[key]

def _set_provider(doc, base_url=None):
    providers = _ensure_table(doc, "model_providers")
    if PREFERRED_PROVIDER not in providers:
        providers[PREFERRED_PROVIDER] = tomlkit.table()
    prov = providers[PREFERRED_PROVIDER]
    defaults = dict(PROVIDER_DEFAULTS["local_lmstudio"])
    if base_url:
        defaults["base_url"] = base_url
    for k, v in defaults.items():
        if k not in prov:
            prov[k] = v
    if base_url:
        prov["base_url"] = base_url

def _add_profile(doc, name, model=None, provider=None):
    profiles = _ensure_table(doc, "profiles")
    if name not in profiles:
        profiles[name] = tomlkit.table()
    prof = profiles[name]
    defaults = dict(PROFILE_DEFAULTS.get(name, PROFILE_DEFAULTS[PREFERRED_PROFILE]))
    if model:
        defaults["model"] = model
    if provider:
        defaults["model_provider"] = provider
    for k, v in defaults.items():
        if k not in prof:
            prof[k] = v
    if model:
        prof["model"] = model
    if provider:
        prof["model_provider"] = provider

CATALOG_ENTRY_TEMPLATE = {
    "qwen/qwen3-coder-next": {
        "slug": "qwen/qwen3-coder-next",
        "display_name": "Qwen 3 Coder Next via LM Studio",
        "description": "Local Qwen model served by LM Studio.",
        "default_reasoning_level": "low",
        "supported_reasoning_levels": [{"effort": "low", "description": "Minimal local-model reasoning control"}],
        "shell_type": "shell_command",
        "visibility": "list",
        "minimal_client_version": "0.0.1",
        "supported_in_api": True,
        "availability_nux": None,
        "upgrade": None,
        "priority": 100,
        "base_instructions": "You are Codex, a coding agent. Use the provided tool-calling interface for tool use.",
        "supports_reasoning_summaries": False,
        "reasoning_summary_format": "none",
        "default_reasoning_summary": "none",
        "support_verbosity": False,
        "default_verbosity": "low",
        "apply_patch_tool_type": "function",
        "input_modalities": ["text"],
        "supports_image_detail_original": False,
        "truncation_policy": {"mode": "tokens", "limit": 100000},
        "supports_parallel_tool_calls": False,
        "context_window": 262144,
        "max_context_window": 262144,
        "auto_compact_token_limit": 240000,
        "effective_context_window_percent": 95,
        "experimental_supported_tools": [],
    },
}

def _apply_to_doc(doc, key, value):
    if key == "set_provider" or key == "set_provider_url":
        _set_provider(doc, base_url=value)
    elif key == "set_wire_api":
        _ensure_table(doc, "model_providers")
        prov = doc["model_providers"].get(PREFERRED_PROVIDER)
        if prov is None:
            _set_provider(doc)
            prov = doc["model_providers"][PREFERRED_PROVIDER]
        prov["wire_api"] = value
    elif key == "add_profile":
        _add_profile(doc, value)
    elif key == "fix_profile_model":
        _add_profile(doc, PREFERRED_PROFILE, model=value)
    elif key == "fix_profile_provider":
        _add_profile(doc, PREFERRED_PROFILE, provider=value)
    elif key == "set_default_profile":
        doc["profile"] = value

planned = [(sev, msg, key) for sev, msg, key in issues if key]
if not planned:
    print("No fixable issues. Nothing to apply.")
else:
    print("Planned fixes:")
    for sev, msg, key in planned:
        print(f"  - [{sev}] {key}: {msg}")

if APPLY and planned:
    # 1. Catalog
    cat_changed = False
    if not CATALOG_FILE.exists():
        CATALOG_DIR.mkdir(parents=True, exist_ok=True)
        catalog = {"models": []}
        cat_changed = True
    for sev, msg, key in planned:
        if key == "add_catalog_entry":
            slug = msg.split("'")[1] if "'" in msg else PREFERRED_MODEL
            entry = CATALOG_ENTRY_TEMPLATE.get(slug)
            if entry and not any(m["slug"] == slug for m in catalog.get("models", [])):
                catalog["models"].append(entry)
                cat_changed = True
    if cat_changed:
        if CATALOG_FILE.exists():
            _backup(CATALOG_FILE)
        CATALOG_FILE.write_text(json.dumps(catalog, indent=2) + "\n")
        print(f"Wrote {CATALOG_FILE}")

    # 2. User config
    if user_doc is None:
        USER_CONFIG.parent.mkdir(parents=True, exist_ok=True)
        user_doc = tomlkit.parse(USER_CONFIG_TEMPLATE.format(profile=PREFERRED_PROFILE))
        _set_provider(user_doc)
        _add_profile(user_doc, PREFERRED_PROFILE)
    else:
        _backup(USER_CONFIG)
    for sev, msg, key in planned:
        if key in {"create_user_config", "create_catalog", "add_catalog_entry"}:
            continue
        # locate the value the diagnostic captured
        fix = fixes.get(key)
        val = fix[1] if isinstance(fix, tuple) and len(fix) > 1 else None
        _apply_to_doc(user_doc, key, val)
    USER_CONFIG.write_text(tomlkit.dumps(user_doc))
    print(f"Wrote {USER_CONFIG}")
    print()
    print("--- new ~/.codex/config.toml ---")
    print(USER_CONFIG.read_text())
elif planned:
    print()
    print("Set APPLY = True and re-run this cell to write the changes.")

No fixable issues. Nothing to apply.


## 5. Verify

Re-run cells 1–3 to confirm the diagnostic is clean, then optionally smoke-test
the active profile with a tiny `codex exec` call. This requires the `codex` CLI
on PATH and the LM Studio model loaded.

In [6]:
import subprocess

codex = env_facts.get("codex_bin")
if not codex:
    print("codex CLI not on PATH — skipping smoke test.")
else:
    try:
        out = subprocess.run(
            [codex, "--version"],
            capture_output=True, text=True, timeout=10,
        )
        print("codex --version:", out.stdout.strip() or out.stderr.strip())
    except (subprocess.TimeoutExpired, OSError) as e:
        print("codex --version failed:", e)

codex --version: codex-cli 0.130.0
